# Setup

In [ ]:
library(data.table)
library(Matrix)     # km.Rdata holds a dsCMatrix; `[` needs Matrix ATTACHED, not just
                    # loaded as a namespace, or the subset below fails with
                    # "object of type 'S4' is not subsettable". 01c loads it for the same reason.
options(datatable.na.strings=c('NA',''))

#dir.create('data/raw/phenotypes',rec=T)
system('gcloud storage cp gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/raw/genomics/dosages.csv data/raw/genomics')

# Join all datasets

In [ ]:
data <-
Reduce(\(x,l) merge(x, l$y, by=l$by, all.x=T),
  list(               x=fread('data/derived/phenotypes/fhs_picsure_variables.csv'),
    list(by='NWD_Id', y=fread('data/raw/genomics/dosages.csv')                                      ),
    list(by='NWD_Id', y=fread('data/raw/phenotypes/freeze9_pcair_results.tsv')[, NWD_Id := sample.id]),
    list(by='TOM_Id', y=fread('data/derived/metabolomics/QCd/merged_QCd-aligned.csv')[grep('FHS',TOM_Id)][,TOM_Id:=sub('.*TOM','TOM',TOM_Id)])) # Merge metabolomics last b/c it adds tons of cols
)

# Rename columns

In [ ]:
data <- data |> setnames(\(nm)  sub('^PC','gPC', nm))# 'PC#' -> 'gPC#' To distinguish genetic PCs from metabolite PCs we'll add later
data <- data |> setnames('SEX','sex')

## Winsorization, imputation, transformation

In [ ]:
winsor      <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) pmin(pmax(v,bounds[1]),bounds[2])
na_outliers <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) fifelse(v<bounds[1] | v>bounds[2], NA, v)
nafill2 <- \(x) c(NA,x[!is.na(x)])[cumsum(!is.na(x))+1] # nafill(type='locf') but can handle non-numeric

data <- data[
  # (Median) imputation of covariates if not present in any exam (i.e. not present in exam 1)
  # (Making sure to take the median of the original set of non-missing values, NOT recalculating the median after each imputation!)
  ][, {age_median <<- median(age, na.rm=T);
       bmi_median <<- median(bmi, na.rm=T);
       .SD}

  ][ is.na(age), age := age_median
  ][ is.na(bmi), bmi := bmi_median

  # Definitions
  ][, hdl_log   := log(hdl)
  ][,  tg_log   := log(tg )
]

## Decision record: sparsifying the FHS kinship matrix

**What we found.** The kinship matrix in `km.Rdata` is raw, not thresholded. In the FHS
block (7,144 x 7,144) **32.4% of entries are non-zero** -- 16.5M of them -- but almost all
are noise: median off-diagonal 1e-4, and values as low as -0.014, which a true kinship
coefficient cannot take. Only ~23,000 pairs (~3.2 per person) exceed 0.05. The diagonal is
~0.5, so this is a kinship coefficient matrix, not a GRM.

**Why it mattered.** `lme4qtl` replaces `Zt` with `R'Zt` where `R = chol(km)`. With a
dense-ish `km` the factor fills in, `Zt` becomes dense, and *every* likelihood evaluation
(~30 per fit) becomes a dense 7,144^2 factorization. Density changes the complexity class,
not just the constant: in benchmarking, going from sparse to dense at n=1,600 cost 541x,
and scaled cubically thereafter. Fits took hours; `04c` appeared to hang.

**What we chose.** Zero all |kinship| < **0.025**, then add a **0.01 ridge** to the
diagonal. From a threshold x ridge sweep on the real matrix:

| threshold | ridge needed for PD | non-zeros | % non-zero |
|---|---|---|---|
| 0.025 | 0.01 | 66,528 | 0.130 |
| 0.05  | 0.05 | 53,200 | 0.104 |
| 0.088 | not PD at any ridge tested | 38,866 | 0.076 |
| 0.125 | not PD at any ridge tested | 32,464 | 0.064 |

0.025 with a 0.01 ridge was the **least aggressive** option that Cholesky accepted. One fit
went from hours to **0.78 s**. Thresholding alone is not positive definite here because a
few near-duplicate pairs (likely MZ twins or repeat samples) have off-diagonal kinship close
to their own diagonal; at higher thresholds those pairs become isolated indefinite blocks,
which is why 0.088 and 0.125 fail outright.

**Implications.**

* *Statistical.* Dropping relatedness below 0.025 discards pairs more distant than about
  fourth-degree, whose contribution is indistinguishable from estimation noise. The ridge
  inflates self-kinship by 2%, which slightly shrinks the genetic variance component; it
  does not bias fixed effects. Both are conventional (cf. sparse-GRM practice in GCTA /
  fastGWA, and the sparse ancestry-adjusted relatedness approach already cited in the
  methods).
* *For the manuscript.* The methods need a sentence: relatedness thresholded at 0.025 with
  a 0.01 diagonal ridge, applied to FHS. Do not leave this implicit -- it is a real analysis
  choice, and the alternative (raw matrix) is computationally infeasible rather than merely
  slow.
* **Open question.** MESA's kinship, drawn from the *same* source file, measured 99.97%
  zero (~0.9 relatives/person) versus FHS's raw 32.4%. Some of that is real -- FHS is a
  family cohort -- but the gap suggests MESA's block may already have been thresholded
  upstream, possibly at a different cutoff. If so the two cohorts currently define
  "related" differently and should be reconciled before publication. Check with:
  `min(abs(km@x[km@x < 0.4]))` on the MESA matrix; if it is ~0.05 or ~0.088, match FHS to it.

**When to revisit.** If the FHS sample set changes, the sweep should be re-run -- the
threshold/ridge pair is specific to this matrix, and the `stopifnot` below will fail rather
than silently fall back to a dense eigendecomposition.

# Remap kinship matrix ids

In [ ]:
dir.create('data/raw/genomics',rec=T)
dir.create('data/derived/genomics',rec=T)

if(!file.exists('data/raw/genomics/km.Rdata')) system('gcloud storage cp gs://fc-secure-c5905035-bf75-471b-8fa8-fb8a7f83c4a9/submissions/e81b3733-44cb-4c17-87db-2aee99a0491a/fetch_dbgap/d64cc7fd-47b7-4352-985f-3c3951b9c3cf/call-decrypt/glob-b3ddbe8d141590fbae0db21546878fa2/km.Rdata data/raw/genomics')
load('data/raw/genomics/km.Rdata')
km <- km[rownames(km) %in% data$NWD_Id,
         colnames(km) %in% data$NWD_Id]

# Sparsify: the source matrix is raw (32% non-zero, almost all noise), which makes
# chol(km) fill in and every likelihood evaluation a dense 7144^2 factorization.
# See the decision record above for the sweep behind these two values.
KM_THRESHOLD <- 0.025
KM_RIDGE     <- 0.01
km_raw <- km
km[abs(km) < KM_THRESHOLD] <- 0
diag(km) <- diag(km_raw) + KM_RIDGE
km <- as(as(as(km,'dMatrix'),'symmetricMatrix'),'CsparseMatrix')
dimnames(km) <- dimnames(km_raw)
stopifnot('sparsified km is not positive definite' =
          !inherits(try(lme4qtl:::relfac.chol(km), silent=TRUE), 'try-error'))
cat('kinship non-zeros:', format(Matrix::nnzero(km_raw), big.mark=','), '->',
    format(Matrix::nnzero(km), big.mark=','),
    '(', round(100*Matrix::nnzero(km)/nrow(km)^2, 3), '% )\n')

save(km, file='data/derived/genomics/fhs_km.RData')

## Write

In [ ]:
fwrite(data, 'data/derived/analysis_df-fhs.csv')

In [ ]:
system('gcloud storage cp data/derived/analysis_df-fhs.csv        gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/analysis_df-fhs.csv')
system('gcloud storage cp data/derived/genomics/fhs_km.RData gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/genomics/fhs_km.RData')